In [1]:
import jax
import jax.numpy as jnp
import numpy as onp
import jax_sn
%load_ext autoreload
%autoreload 2
jax.config.update("jax_enable_x64", True)

In [2]:
from jax_sn.tools.setup_test_mgop import setup_slab_mgop, make_xs_test, setup_layered_slab_mgop

In [3]:

def _build_base_mgop():
    """Build the reference mgop with default (Vmap) operators. Expensive — call once."""
    xs = [make_xs_test(m, n_groups=2, l_order=1) for m in ["fuel", "moderator", "reflector"]]
    return setup_layered_slab_mgop(
        dim=3,
        degree=1,
        tn_quadrature_set=3,
        region_sizes=[1.0, 4.0, 2.0],
        region_resolutions=[1330, 2,1],
        region_xs=xs,
        region_sources=[1.0, 0.5, 0.2],
        transverse_specs=[(2.0, 2), (1.523, 3)],
    )



In [4]:



onp.random.seed(2042026)

base_mgop = _build_base_mgop()

b_apply = jnp.array(onp.random.random_sample(base_mgop.transport_operator.out_structure().shape))
xs = base_mgop.domain.cross_sections_indexed.map_to_full_array(0)


In [27]:
import equinox as eqx

# @eqx.filter_jit
# def sweep_sequential_sa_with_warp(top,sigma_t, x_arr, a : int):    

#     args_flat_ta = _flatten_transport_operator(top)
#     print(args_flat_ta[-6])
#     return sweep_sa_jax(a, *args_flat_ta, sigma_t, x_arr, output_dims = {"result" :  x_arr.shape})[0]

from jax_sn.operators.transport_operator.linear_tetrahedron_warp_kernel import _sweep_sequential_sa_with_warp, _sweep_sequential_ma_with_warp, TransportOperatorSequentialWarpSingleAngle


_sweep_sequential_sa_with_warp(base_mgop.transport_operator[0], b_apply[0])

Array([[ 2.18076280e+04,  2.18025772e+04,  2.12541226e+04,
         2.25709544e+04],
       [ 2.18089978e+04,  2.12716133e+04,  2.25631130e+04,
         2.20207166e+04],
       [ 2.17991357e+04,  2.25853758e+04,  2.20228682e+04,
         2.25691485e+04],
       ...,
       [ 2.57375517e+01,  6.68755718e+00, -4.98747211e+00,
        -4.30259044e+00],
       [ 2.46117183e+01,  4.68976625e+00,  9.34396744e+00,
         5.78231461e+00],
       [ 1.13872592e+01, -6.16430884e+00,  1.62272705e+01,
        -4.74404640e+00]], dtype=float64)

In [18]:
%timeit _sweep_sequential_sa_with_warp(base_mgop.transport_operator[0], b_apply[0])

536 ms ± 8.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [20]:
%timeit _sweep_sequential_ma_with_warp(base_mgop.transport_operator, b_apply).block_until_ready()

666 ms ± 1.53 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [30]:
top_2 = TransportOperatorSequentialWarpSingleAngle(base_mgop.solution_domain_data, base_mgop.transport_operator[0].topological_sorted_angle, 0)

In [35]:
@eqx.filter_jit
def aaa(top_2 : TransportOperatorSequentialWarpSingleAngle):
    return jnp.sum(top_2.inv(jnp.ones_like(top_2.out_structure())))


aaa(top_2).block_until_ready()

Array(4.54339681e+09, dtype=float64)

In [40]:
grad_f = eqx.filter_jit(eqx.filter_grad(aaa))

In [47]:
%timeit grad_f(top_2).solution_domain_data.precomputed_geometric_data.vol_jacobians_inv.block_until_ready()

1.04 s ± 3.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [48]:
%timeit grad_f(base_mgop.transport_operator[0]).solution_domain_data.precomputed_geometric_data.vol_jacobians_inv.block_until_ready()

1.09 s ± 11.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [44]:
from jax_sn.tools import assert_pytree_allclose

assert_pytree_allclose(grad_f(top_2).solution_domain_data, grad_f(base_mgop.transport_operator[0]).solution_domain_data)

AssertionError: Pytree mismatches found:
(GetAttrKey(name='domain'), GetAttrKey(name='face_normals')): 
Not equal to tolerance rtol=1e-12, atol=1e-12

Mismatched elements: 80 / 575856 (0.0139%)
First 5 mismatches are at indices:
 [4, 3, 0]: 16.154232606764353 (ACTUAL), 16.154232606810464 (DESIRED)
 [4, 3, 1]: 2.3077475152520504 (ACTUAL), 2.3077475152586375 (DESIRED)
 [4, 3, 2]: 2.307747515252051 (ACTUAL), 2.307747515258638 (DESIRED)
 [5, 3, 0]: 16.242978539672837 (ACTUAL), 16.24297853969984 (DESIRED)
 [5, 3, 1]: 2.320425505667548 (ACTUAL), 2.3204255056714054 (DESIRED)
Max absolute difference among violations: 1.04319042e-08
Max relative difference among violations: 4.65196939e-11
 ACTUAL: array([[[5.260459e+05, 7.514941e+04, 7.514941e+04],
        [3.511287e+05, 5.016124e+04, 5.016124e+04],
        [5.208455e+02, 7.440651e+01, 7.440651e+01],...
 DESIRED: array([[[5.260459e+05, 7.514941e+04, 7.514941e+04],
        [3.511287e+05, 5.016124e+04, 5.016124e+04],
        [5.208455e+02, 7.4406

In [32]:
%timeit top_2.inv(b_apply[0]).block_until_ready()

525 ms ± 921 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [31]:
onp.testing.assert_allclose(
    top_2.inv(b_apply[0]),
    base_mgop.transport_operator[0].inv(b_apply[0]),
    rtol=1e-5,
    atol=1e-5,)

In [25]:
onp.testing.assert_allclose(
    _sweep_sequential_ma_with_warp(base_mgop.transport_operator, b_apply),
    base_mgop.transport_operator.inv(b_apply),
    rtol=1e-5,
    atol=1e-5,)

In [ ]:
from warp.jax_experimental import jax_callable

jax_callable()

In [ ]:
result_buffer = jnp.zeros_like(b_apply[0])

In [ ]:
ws.single_angle(0, xs.total, b_apply[0], result_buffer)

In [ ]:
result_buffer_ma = jnp.zeros_like(b_apply)

In [ ]:
import warp as wp
reee = (ws.multi_angle(xs.total, b_apply, result_buffer_ma))

In [ ]:
%timeit -n 10 -r 3 ws.single_angle(0, xs.total, b_apply[0], result_buffer)

523 ms ± 382 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)


In [ ]:
%timeit -n 5 -r 3 ws.multi_angle(xs.total, b_apply, result_buffer_ma)

671 ms ± 10.5 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)


In [ ]:
%timeit  base_mgop.transport_operator.inv(b_apply).block_until_ready()

709 ms ± 4.18 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
onp.testing.assert_allclose(result, base_mgop.transport_operator[0].inv(b_apply[0]))

In [ ]:
onp.testing.assert_allclose(result_buffer_ma, base_mgop.transport_operator.inv(b_apply))

AssertionError: 
Not equal to tolerance rtol=1e-07, atol=0

Mismatched elements: 1 / 4188672 (2.39e-05%)
Mismatch at index:
 [58, 12959, 0]: -0.00039902634642562795 (ACTUAL), -0.0003990261440360854 (DESIRED)
Max absolute difference among violations: 2.02389543e-10
Max relative difference among violations: 5.07208727e-07
 ACTUAL: array([[[ 6.439454e+03,  6.434124e+03,  6.379089e+03,  6.690254e+03],
        [ 6.440841e+03,  6.396528e+03,  6.682499e+03,  6.633334e+03],
        [ 6.430937e+03,  6.705093e+03,  6.635500e+03,  6.688554e+03],...
 DESIRED: array([[[ 6.439454e+03,  6.434124e+03,  6.379089e+03,  6.690254e+03],
        [ 6.440841e+03,  6.396528e+03,  6.682499e+03,  6.633334e+03],
        [ 6.430937e+03,  6.705093e+03,  6.635500e+03,  6.688554e+03],...

In [ ]:
import pyvista as pv
from jax_sbgeom.jax_utils import mesh_to_pyvista_mesh
plotter  = pv.Plotter()
plotter.add_mesh(mesh_to_pyvista_mesh(mgop.domain.vertices, mgop.domain.connectivity), scalars = mgop.domain.cross_sections_indexed.indices)
plotter.show()

Widget(value='<iframe src="http://localhost:44001/index.html?ui=P_0xe874841174a0_5&reconnect=auto" class="pyvi…